In [ ]:
import sys
!"{sys.executable}" -m pip install polars pyarrow

In [ ]:
import polars as pl


df1 = pl.read_csv(r"E:\Data Analyst track with DEPI\the project\MH-CLD\2021\mhcld_puf_2021.csv", columns=["CASEID"])
df2 = pl.read_csv(r"E:\Data Analyst track with DEPI\the project\MH-CLD\2022\mhcld_puf_2022.csv", columns=["CASEID"])


overlap = df1.join(df2, on="CASEID", how="inner")
print(f"عدد الأرقام المتكررة بين السنتين: {len(overlap)}")

عدد الأرقام المتكررة بين السنتين: 0


In [ ]:
import polars as pl

# ==========================================
# 1. Define File Paths (Inputs and Output)
# ==========================================
file_2021 = r"E:\Data Analyst track with DEPI\the project\MH-CLD\2021\mhcld_puf_2021.csv"
file_2022 = r"E:\Data Analyst track with DEPI\the project\MH-CLD\2022\mhcld_puf_2022.csv"
file_2023 = r"E:\Data Analyst track with DEPI\the project\MH-CLD\2023\mhcld_puf_2023.csv"

output_path = r"E:\Data Analyst track with DEPI\the project\MH-CLD\MH_CLD_2021_2023_Cleaned.parquet"

# ==========================================
# 2. Feature Selection (Columns to keep)
# ==========================================
columns_to_keep = [
    "YEAR", "CASEID", "STATEFIP",
    "AGE", "SEX", "RACE", "ETHNIC", "EDUC", "MARSTAT", "EMPLOY", "LIVARAG", "VETERAN",
    "SMISED", "MH1", "MH2", "SUB",
    "SPHSERVICE", "CMPSERVICE", "OPISERVICE", "RTCSERVICE"
]

print("Starting the data processing pipeline...")

# ==========================================
# 3. Lazy Loading & Concatenation
# ==========================================
print("Reading CSV files and selecting columns...")
df_2021 = pl.scan_csv(file_2021, ignore_errors=True).select(columns_to_keep)
df_2022 = pl.scan_csv(file_2022, ignore_errors=True).select(columns_to_keep)
df_2023 = pl.scan_csv(file_2023, ignore_errors=True).select(columns_to_keep)

# Combine the three years
combined_lazy_df = pl.concat([df_2021, df_2022, df_2023])

# ==========================================
# 4. Preprocessing (Transform -9 to Null)
# ==========================================
print("Handling missing values (-9 to Null)...")
cleaned_lazy_df = combined_lazy_df.with_columns(
    
    pl.when(pl.all() == -9).then(None).otherwise(pl.all()).name.keep()
)

# ==========================================
# 5. Execution & Saving
# ==========================================
print("Executing and compressing to Parquet...")
cleaned_lazy_df.sink_parquet(output_path)

print("Pipeline executed successfully!")
print(f"File saved at: {output_path}")

Starting the data processing pipeline...
Reading CSV files and selecting columns...
Handling missing values (-9 to Null)...
Executing and compressing to Parquet...
Pipeline executed successfully!
File saved at: E:\Data Analyst track with DEPI\the project\MH-CLD\MH_CLD_2021_2023_Cleaned.parquet


In [1]:
import polars as pl

In [ ]:
import polars as pl

file_path = r"E:\Data Analyst track with DEPI\the project\MH-CLD\MH_CLD_2021_2023_Cleaned.parquet"


df_lazy = pl.scan_parquet(file_path)


total_rows = df_lazy.select(pl.len()).collect().item()
print(f"إجمالي عدد الحالات في الملف: {total_rows:,} مريض")


year_counts = df_lazy.group_by("YEAR").len().sort("YEAR").collect()
print("\n--- الأعداد مقسمة حسب كل سنة ---")
print(year_counts)

إجمالي عدد الحالات في الملف: 20,569,570 مريض

--- الأعداد مقسمة حسب كل سنة ---
shape: (3, 2)
┌──────┬─────────┐
│ YEAR ┆ len     │
│ ---  ┆ ---     │
│ i64  ┆ u32     │
╞══════╪═════════╡
│ 2021 ┆ 6542630 │
│ 2022 ┆ 6991299 │
│ 2023 ┆ 7035641 │
└──────┴─────────┘


In [3]:
print(df_lazy.schema)

Schema([('YEAR', Int64), ('CASEID', Int64), ('STATEFIP', Int64), ('AGE', Int64), ('SEX', Int64), ('RACE', Int64), ('ETHNIC', Int64), ('EDUC', Int64), ('MARSTAT', Int64), ('EMPLOY', Int64), ('LIVARAG', Int64), ('VETERAN', Int64), ('SMISED', Int64), ('MH1', Int64), ('MH2', Int64), ('SUB', Int64), ('SPHSERVICE', Int64), ('CMPSERVICE', Int64), ('OPISERVICE', Int64), ('RTCSERVICE', Int64)])


C:\Users\EL3ATTY\AppData\Local\Temp\ipykernel_5144\2480285578.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(df_lazy.schema)


In [6]:
df_lazy.head().collect()

YEAR,CASEID,STATEFIP,AGE,SEX,RACE,ETHNIC,EDUC,MARSTAT,EMPLOY,LIVARAG,VETERAN,SMISED,MH1,MH2,SUB,SPHSERVICE,CMPSERVICE,OPISERVICE,RTCSERVICE
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2021,20210000001,1,7,2,5,2,4,1,null,3,2,1,11,null,null,2,1,1,2
2021,20210000002,1,4,2,5,4,4,1,null,2,null,1,7,2,null,1,2,2,2
2021,20210000003,1,11,1,3,3,5,1,5,3,1,1,11,13,5,1,1,2,2
2021,20210000004,1,7,2,3,3,null,1,null,null,null,1,11,null,8,2,1,1,2
2021,20210000005,1,3,2,5,4,3,1,5,2,null,2,2,13,null,1,1,2,2
